In [ ]:
from stable_whisper import WhisperResult

pre_merge_min_gap = 0.5
input_folder_name = "../data/knesset/committee/"
session_id = "2074447"
transcript_in_filename = input_folder_name + session_id + "/transcript.aligned.json"
audio_file = input_folder_name + session_id + "/audio.m4a"
result = WhisperResult(transcript_in_filename)#.merge_by_gap(pre_merge_min_gap, max_words=25)

In [ ]:
import sys
sys.path.insert(0, '..')

from vad.vad_io import load_frame_vad_probs
from vad.definitions import SPEECH_PROB_FRAME_DURATION

vad_probs_filename = input_folder_name + session_id + '/speech_probs.frame'
speech_probs = load_frame_vad_probs(vad_probs_filename)

print(f'Loaded {len(speech_probs)} VAD frames, '
      f'covering {len(speech_probs) * SPEECH_PROB_FRAME_DURATION:.1f}s '
      f'at {SPEECH_PROB_FRAME_DURATION*1000:.0f}ms per frame')

In [ ]:
import numpy as np
from scipy.signal import medfilt
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from sources.knesset.committee.refine_segments import (
    smooth_speech_probs,
    adjust_gap,
    DEFAULT_MIN_GAP_TO_ADJUST,
)

# Parameters — use the same defaults as refine_segments.py
MIN_GAP_TO_ADJUST = DEFAULT_MIN_GAP_TO_ADJUST
VAD_WINDOW_RADIUS_SEC = 1.0  # seconds; used only for the plot window
MED_FILTER_KERNEL_SIZE = 5

# Pre-smooth speech_probs once — passed into adjust_gap for every segment pair
smoothed_probs = smooth_speech_probs(speech_probs)


def extract_vad_window(
    speech_probs: np.ndarray,
    center_start: float,
    center_end: float,
    radius_sec: float = VAD_WINDOW_RADIUS_SEC,
) -> tuple[np.ndarray, float, float]:
    """Return (window_frames, window_start_sec, window_end_sec)."""
    total_frames = len(speech_probs)
    window_start_sec = max(0.0, center_start - radius_sec)
    window_end_sec = min(
        total_frames * SPEECH_PROB_FRAME_DURATION,
        center_end + radius_sec,
    )
    frame_start = int(window_start_sec / SPEECH_PROB_FRAME_DURATION)
    frame_end = min(int(window_end_sec / SPEECH_PROB_FRAME_DURATION), total_frames)
    return speech_probs[frame_start:frame_end], window_start_sec, window_end_sec


def plot_gap(
    curr,
    nxt,
    speech_probs: np.ndarray,
    new_curr_end: float | None = None,
    new_next_start: float | None = None,
) -> None:
    """Plot the VAD window around the gap between curr and nxt.

    Vertical lines:
      green  — curr.end  (original)
      red    — next.start (original)
      dashed green — new_curr_end  (after adjust_gap, if changed)
      dashed red   — new_next_start (after adjust_gap, if changed)
    """
    window, win_start, win_end = extract_vad_window(
        speech_probs, curr.end, nxt.start
    )
    window_filt = medfilt(window, kernel_size=MED_FILTER_KERNEL_SIZE)
    times = win_start + np.arange(len(window)) * SPEECH_PROB_FRAME_DURATION

    fig, ax = plt.subplots(figsize=(14, 3))
    ax.fill_between(times, window, alpha=0.4, label='speech prob')
    ax.plot(times, window, linewidth=0.8)
    ax.plot(times, window_filt, linewidth=0.6)

    ax.axvline(curr.end, color='green', linewidth=1.5, label=f'curr.end ({curr.end:.3f}s)')
    ax.axvline(nxt.start, color='red', linewidth=1.5, label=f'next.start ({nxt.start:.3f}s)')

    if new_curr_end is not None and new_curr_end != curr.end:
        ax.axvline(new_curr_end, color='green', linewidth=1.5, linestyle='--',
                   label=f'new curr.end ({new_curr_end:.3f}s)')
    if new_next_start is not None and new_next_start != nxt.start:
        ax.axvline(new_next_start, color='red', linewidth=1.5, linestyle='--',
                   label=f'new next.start ({new_next_start:.3f}s)')

    ax.set_xlim(win_start, win_end)
    ax.set_ylim(0.0, 1.01)
    ax.set_xlabel('time (s)')
    ax.set_ylabel('speech prob')
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(0.1))
    ax.grid(axis='x', which='both', alpha=0.3)
    ax.legend(fontsize=8)
    ax.set_title(
        f'Segments {curr.id} → {nxt.id} | '
        f'gap {nxt.start - curr.end:.3f}s | '
        f'curr: "{curr.text.strip()[:40]}" | '
        f'next: "{nxt.text.strip()[:40]}"'
    )
    plt.tight_layout()
    plt.show()


# ---- pick a segment pair to explore ----
# examples (pre 0.5 gap merge): segment 71 - end too soon, large gap | 78 - start too late, small gap to 77
# examples (post 0.5 gap merge): seg 10 end on time, 11 starts too late | seg 126 end too soon, start too late, very high speech prob in between, 
curr_segment_id = 127   # index into result.segments; (curr, next) = [id], [id+1]

segments = list(result.segments)
curr = segments[curr_segment_id]
nxt  = segments[curr_segment_id + 1]

gap = nxt.start - curr.end
print(f'curr [{curr.id}]: end={curr.end:.3f}s  "{curr.text.strip()}"')
print(f'next [{nxt.id}]: start={nxt.start:.3f}s  "{nxt.text.strip()}"')
print(f'gap: {gap:.3f}s  (adjust threshold: {MIN_GAP_TO_ADJUST}s)')

if gap > MIN_GAP_TO_ADJUST:
    new_curr_end, new_next_start = adjust_gap(curr.end, nxt.start, smoothed_probs)
    plot_gap(curr, nxt, speech_probs, new_curr_end, new_next_start)
else:
    print('Gap is below threshold — no adjustment attempted.')
    plot_gap(curr, nxt, speech_probs)

In [ ]:
import random
from pydub import AudioSegment
from IPython.display import display, Audio, HTML

AUDIO_CONTEXT_SEC = 2.0  # seconds of context around each boundary

# Load the audio file once
audio_path = input_folder_name + session_id + '/audio.m4a'
audio = AudioSegment.from_file(audio_path)


def slice_audio(audio: AudioSegment, start_sec: float, end_sec: float) -> Audio:
    """Return an IPython Audio widget for audio[start_sec:end_sec]."""
    start_ms = max(0, int(start_sec * 1000))
    end_ms   = min(len(audio), int(end_sec * 1000))
    chunk = audio[start_ms:end_ms]
    return Audio(data=chunk.export(format='wav').read(), rate=chunk.frame_rate)


# Browse all pairs with a gap above the threshold
eligible_pairs = [
    (segments[i], segments[i + 1])
    for i in range(len(segments) - 1)
    if segments[i + 1].start - segments[i].end > MIN_GAP_TO_ADJUST
]
print(f'{len(eligible_pairs)} eligible pairs (gap > {MIN_GAP_TO_ADJUST}s) out of {len(segments) - 1} total')

# Plot + play N random samples with adjust_gap applied
left_to_show = 10
# random.shuffle(eligible_pairs)
for c, n in eligible_pairs[0:]:
    new_curr_end, new_next_start = adjust_gap(c.end, n.start, smoothed_probs)

    # skip if not changed
    # if new_curr_end == c.end and new_next_start == n.start:
    #     continue

    left_to_show -= 1
    if left_to_show == 0:
        break

    # --- text header -----------------------------------------------------
    display(HTML(
        f'<hr/>'
        f'<b>curr [{c.id}]:</b> {c.text.strip()}<br/>'
        f'<b>next [{n.id}]:</b> {n.text.strip()}'
    ))

    # --- VAD plot --------------------------------------------------------
    plot_gap(c, n, speech_probs, new_curr_end, new_next_start)

    # --- audio clips -----------------------------------------------------
    display(HTML('<b>curr tail — original</b> &nbsp; [end−2s … end]'))
    display(slice_audio(audio, c.end - AUDIO_CONTEXT_SEC, c.end))

    display(HTML('<b>curr tail — adjusted</b> &nbsp; [end−2s … new_end]'))
    display(slice_audio(audio, c.end - AUDIO_CONTEXT_SEC, new_curr_end))

    display(HTML('<b>next head — original</b> &nbsp; [start … start+2s]'))
    display(slice_audio(audio, n.start, n.start + AUDIO_CONTEXT_SEC))

    display(HTML('<b>next head — adjusted</b> &nbsp; [new_start … start+2s]'))
    display(slice_audio(audio, new_next_start, n.start + AUDIO_CONTEXT_SEC))
